# Task 4 — Guarded bilingual clinical report generation

This notebook is for the **next phase, after transcript and speaker-role verification**. It uses the recovered Task 3 fine-tuned Qwen2.5-1.5B adapter in 4-bit mode and produces an English and source-language version of the same 12-field report.

The notebook refuses unverified rows and skips inputs that exceed the configured context instead of silently truncating them. It records prompt/output tokens and GPU usage. Model output remains a draft: validate every clinical fact, medicine, number, and translation before presenting it as a report.

**Do not upload real patient identifiers or clinical records to a personal Colab.** The current project audio uses a fictional scripted consultation.


## 1. Prepare the input files

Create `verified_transcripts.csv` with these columns:

`sample_id,language,verified_transcript,transcript_status,transcript_reviewer,speaker_roles_verified`

Use one row per recording. Set `transcript_status` to exactly `verified` only after a fluent reviewer checks the full transcript against the audio. Include verified speaker labels such as `[DOCTOR]`, `[PATIENT]`, and `[NURSE]` in the transcript. Set `speaker_roles_verified` to `yes` only after listening and checking all roles.

Zip the existing Task 3 `final_adapter` directory into `final_adapter.zip`. Do not upload secrets or Hugging Face tokens.


In [ ]:
!pip -q install unsloth pandas

import gc, json, time, zipfile
from pathlib import Path
import pandas as pd
import torch
from google.colab import files
from unsloth import FastLanguageModel
from peft import PeftModel

assert torch.cuda.is_available(), "Select a Colab GPU runtime (T4 is sufficient for this 4-bit 1.5B model)."
print("GPU:", torch.cuda.get_device_name(0))


## 2. Upload transcript and adapter

Upload the verified transcript CSV and `final_adapter.zip`. If the reference CSV has any unverified transcript or unverified speaker roles, the next cell will stop rather than generating a report.


In [ ]:
uploaded = files.upload()
required = {"verified_transcripts.csv", "final_adapter.zip"}
missing = required - set(uploaded)
assert not missing, f"Missing required file(s): {sorted(missing)}"

transcripts_path = Path("verified_transcripts.csv")
adapter_zip = Path("final_adapter.zip")
ADAPTER_DIR = Path("/content/final_adapter")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(adapter_zip) as zf:
    zf.extractall(ADAPTER_DIR)
# Some zips contain a top-level final_adapter/ directory. Locate adapter_config.json.
config_files = list(ADAPTER_DIR.rglob("adapter_config.json"))
assert config_files, "adapter_config.json was not found inside final_adapter.zip"
ADAPTER_PATH = config_files[0].parent
print("Adapter found at:", ADAPTER_PATH)

df = pd.read_csv(transcripts_path).fillna("")
required_cols = {"sample_id", "language", "verified_transcript", "transcript_status", "transcript_reviewer", "speaker_roles_verified"}
assert required_cols.issubset(df.columns), f"CSV needs columns: {sorted(required_cols)}"
assert len(df), "No transcript rows were uploaded."
assert (df["transcript_status"].str.strip().str.lower() == "verified").all(), "Every transcript must be marked verified."
assert df["transcript_reviewer"].str.strip().ne("").all(), "Every transcript needs a fluent reviewer name."
assert df["speaker_roles_verified"].str.strip().str.lower().eq("yes").all(), "Verify all speaker roles before report extraction."
assert df["verified_transcript"].str.strip().ne("").all(), "A verified transcript is blank."
assert df["sample_id"].is_unique, "sample_id values must be unique."
print("Verified rows ready:", len(df))


## 3. Load the matching fine-tuned baseline

Baseline recovery showed that the adapter matches Qwen2.5-1.5B-Instruct, not Qwen2.5-7B. The downloaded adapter metadata identifies this exact base. This run uses that ID and does not retrain it. If the adapter load fails, stop and check that the adapter came from that same base model.


In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 640  # both language versions + evidence; reduce only if memory/time requires
OUTPUT_DIR = Path("/content/task4_guarded_reports")
OUTPUT_DIR.mkdir(exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_INPUT_TOKENS,
    dtype=None, load_in_4bit=True
)
model = PeftModel.from_pretrained(model, str(ADAPTER_PATH), is_trainable=False)
FastLanguageModel.for_inference(model)
print("Loaded exact Task 3 base and LoRA adapter in 4-bit mode:", BASE_MODEL)


## 4. Generate auditable English and source-language drafts

Each report uses the same 12 fields. The `source_evidence` values quote the verified transcript to help a human reviewer audit claims. The model is instructed to use `Not stated` / a source-language equivalent or `Unclear` rather than infer absent information. Evidence quotes do not prove correctness; review them against the transcript.


In [ ]:
REPORT_FIELDS = [
    "patient_information", "chief_complaint", "symptoms",
    "history_of_present_illness", "past_medical_history", "family_history",
    "examination_and_labs", "diagnosis_as_stated", "medications", "plan",
    "follow_up", "warning_signs",
]
LANGUAGES = {"Arabic", "Bengali", "Malayalam", "Urdu"}

def build_prompt(language, transcript):
    schema = {
        "english_report": {k: "string; use Not stated or Unclear when appropriate" for k in REPORT_FIELDS},
        "source_language_report": {k: "string in the same language and native script as the conversation" for k in REPORT_FIELDS},
        "source_evidence": {k: "short exact quote from the verified source transcript, or empty string if not stated" for k in REPORT_FIELDS},
    }
    return f"""You are formatting a verified doctor-patient-nurse conversation into a draft clinical summary.
Rules: use only explicitly spoken facts; do not infer a diagnosis from symptoms or laboratory values; do not invent, correct, or complete a medicine name, dose, value, date, or patient detail. Keep diagnoses under diagnosis_as_stated only when a clinician explicitly states them. Keep patient-reported symptoms separate from the clinician's assessment. If missing, use Not stated (or a faithful equivalent in the source-language report). If ambiguous, use Unclear. Preserve units and numbers exactly. Do not treat the transcript itself as medical advice.
Return one valid JSON object, with exactly these top-level keys and field names: {json.dumps(schema, ensure_ascii=False)}
The source-language report must be in {language} using its usual native script. English and source-language reports must contain the same facts. source_evidence must quote only the verified source transcript; do not translate evidence quotes.
Verified conversation:
{transcript}
"""

rows=[]
for row in df.to_dict("records"):
    language = str(row["language"]).strip().title()
    if language not in LANGUAGES:
        rows.append({"sample_id":row["sample_id"],"language":language,"status":"unsupported_language"})
        continue
    messages=[{"role":"user","content":build_prompt(language, str(row["verified_transcript"]))}]
    prompt=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc=tokenizer(prompt, return_tensors="pt", truncation=False)
    input_tokens=int(enc["input_ids"].shape[-1])
    if input_tokens > MAX_INPUT_TOKENS:
        rows.append({"sample_id":row["sample_id"],"language":language,"status":"input_too_long_skipped","prompt_tokens":input_tokens})
        continue
    enc={k:v.to(model.device) for k,v in enc.items()}
    torch.cuda.reset_peak_memory_stats()
    started=time.perf_counter()
    with torch.inference_mode():
        generated=model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.eos_token_id)
    seconds=time.perf_counter()-started
    out_ids=generated[0, input_tokens:]
    raw=tokenizer.decode(out_ids, skip_special_tokens=True).strip()
    try:
        parsed=json.loads(raw)
        expected={"english_report","source_language_report","source_evidence"}
        valid=(set(parsed)==expected and all(set(parsed[k])==set(REPORT_FIELDS) for k in expected))
        status="draft_ready" if valid else "invalid_json_schema_review_required"
    except Exception:
        parsed={"raw_output":raw}
        status="invalid_json_review_required"
    rows.append({
        "sample_id":row["sample_id"],"language":language,"status":status,
        "transcript_reviewer":row["transcript_reviewer"],
        "english_report":json.dumps(parsed.get("english_report",{}),ensure_ascii=False),
        "source_language_report":json.dumps(parsed.get("source_language_report",{}),ensure_ascii=False),
        "source_evidence":json.dumps(parsed.get("source_evidence",{}),ensure_ascii=False),
        "raw_output":raw,"prompt_tokens":input_tokens,"output_tokens":int(out_ids.numel()),
        "generation_seconds":round(seconds,2),
        "peak_gpu_gib":round(torch.cuda.max_memory_allocated()/1024**3,2),
    })
    del enc, generated, out_ids
    gc.collect(); torch.cuda.empty_cache()
    print(row["sample_id"], status, "input/output tokens:", input_tokens, len(tokenizer.encode(raw)), "seconds:", round(seconds,2))

results=pd.DataFrame(rows)
result_path=OUTPUT_DIR/"task4_guarded_bilingual_report_drafts.csv"
results.to_csv(result_path,index=False,encoding="utf-8-sig")
print("Saved draft outputs:",result_path)


## 5. Download and review

Rows marked `invalid_json...` need correction or a rerun; do not treat them as completed reports. For `draft_ready`, review source evidence against the verified transcript and then have fluent/clinical reviewers check both report languages, all 12 fields, speaker attribution, names, diagnoses, medicines, values, and units. This small evaluation set does not establish clinical safety or accuracy.


In [ ]:
files.download(str(result_path))
